# Anthropic Batch Workflow

This notebook covers:
1. Generate `requests.json`
2. Create a message batch
3. Poll until `processing_status == "ended"`
4. Download JSONL results
5. Parse them back into the standard artifacts


In [ ]:
from pathlib import Path
import json
import subprocess
import time

from anthropic import Anthropic


In [ ]:
INPUT_POOL = Path('output/05_all_concepts.txt')
BATCH_DIR = Path('output/anthropic_batch')
REQUESTS_PATH = BATCH_DIR / 'requests.json'
MANIFEST_PATH = BATCH_DIR / 'requests_manifest.json'
RESULTS_PATH = BATCH_DIR / 'results.jsonl'
BATCH_DIR.mkdir(parents=True, exist_ok=True)
api_key = 'REPLACE_WITH_API_KEY'
client = Anthropic(api_key=api_key)


In [ ]:
subprocess.run([
    'python', 'prepare_anthropic_batch_requests.py',
    '--input', str(INPUT_POOL),
    '--output', str(BATCH_DIR),
], check=True)

REQUESTS_PATH, MANIFEST_PATH


In [ ]:
with open(REQUESTS_PATH, 'r', encoding='utf-8') as f:
    payload = json.load(f)

len(payload['requests'])


In [ ]:
message_batch = client.messages.batches.create(requests=payload['requests'])
message_batch.id, message_batch.processing_status


In [ ]:
batch_id_path = BATCH_DIR / 'batch_id.txt'
batch_id_path.write_text(message_batch.id + '\n', encoding='utf-8')
batch_id_path


In [ ]:
batch_id = batch_id_path.read_text(encoding='utf-8').strip()
message_batch = client.messages.batches.retrieve(batch_id)
message_batch.id, message_batch.processing_status


In [ ]:
while message_batch.processing_status != 'ended':
    print(message_batch.processing_status, message_batch.request_counts)
    time.sleep(30)
    message_batch = client.messages.batches.retrieve(batch_id)

message_batch.processing_status, message_batch.request_counts


In [ ]:
stream = client.messages.batches.results(batch_id)
with open(RESULTS_PATH, 'w', encoding='utf-8') as f:
    for entry in stream:
        f.write(json.dumps(entry.to_dict(), ensure_ascii=False) + '\n')

RESULTS_PATH


In [ ]:
subprocess.run([
    'python', 'parse_anthropic_batch_results.py',
    '--results', str(RESULTS_PATH),
    '--manifest', str(MANIFEST_PATH),
    '--output', 'output/anthropic',
], check=True)
